# Polish Hate Speech Model Audit — minimal starter notebook

This notebook is a clean foundation for the first stage of the project.

Current scope:
1. install and import the required libraries,
2. load the Polish HateCheck CSV file,
3. load the `ptaszynski/bert-base-polish-cyberbullying` tokenizer and model,
4. inspect the model label mapping,
5. run a tiny smoke test to confirm that inference works.

The evaluation code, metric tables, functionality-level analysis, and counterfactual name-prefix experiment should be added later, once the basic setup runs correctly.

## 0. Colab setup

In Google Colab, first choose:

**Runtime → Change runtime type → GPU**

Then run the installation cell below.

In [ ]:
!pip -q install transformers accelerate torch pandas numpy tqdm

## 1. Imports and configuration

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "ptaszynski/bert-base-polish-cyberbullying"
DATA_PATH = Path("test.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Load the test data

Upload `test.csv` to the Colab session files before running this cell.

Expected useful columns include:
- `test_case` — the Polish text example,
- `label_gold` — the gold HateCheck label,
- `functionality` — the diagnostic category.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find test.csv. Upload it to the Colab session files or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows from {DATA_PATH}")
print("Columns:", list(df.columns))

df.head()

## 3. Load the tokenizer and model

We only need the tokenizer and `pytorch_model.bin` for inference. Do not manually load `training_args.bin`; it is not needed here.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Model loaded successfully.")
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

## 4. Minimal inference helper

This helper returns raw class probabilities from the model. Binary mapping to `harmful` / `non-harmful` should be added only after checking `model.config.id2label`.

In [ ]:
@torch.no_grad()
def predict_proba(texts, batch_size=16, max_length=256):
    """Return model probabilities for a list of texts."""
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)

        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.cpu())

    return torch.cat(all_probs, dim=0).numpy()

## 5. Smoke test

Run inference on a few examples to confirm that the model works end-to-end.

In [ ]:
TEXT_COLUMN = "test_case"

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"Expected a '{TEXT_COLUMN}' column. Available columns: {list(df.columns)}")

sample_texts = df[TEXT_COLUMN].dropna().astype(str).head(5).tolist()
probs = predict_proba(sample_texts, batch_size=5)

label_names = [model.config.id2label[i] for i in range(model.config.num_labels)]
probs_df = pd.DataFrame(probs, columns=label_names)
probs_df.insert(0, "text", sample_texts)

probs_df

## 6. Next steps

Once this notebook runs correctly, the next notebook cells should add:

1. a verified mapping from the model labels to binary `harmful` / `non_harmful`,
2. predictions for the full HateCheck file,
3. overall metrics,
4. metrics grouped by `functionality`,
5. saved CSV files in a `results/` directory,
6. the counterfactual name-prefix experiment.